# 03 · The rehearsal on training data (2019, and 2018)

**In the story** (notebook 00, §3, §4.4 and §6): before 2020 was opened, a rehearsal on 2018–2019 and a dry run of the test script on 2019 checked the code and the model's form. The quadratic in delta and the 1/√T maturity scaling were checked here first, and the walk-forward repeated those checks later (notebooks 08 and 09). This notebook also shows why the Gain intervals resample sessions, not options.

The plan (`analysis_plan.md` §10) runs the whole pipeline on training years before the 2020 test year is opened. This checks the code end to end, previews the width of the confidence intervals, and answers every question about the *form* of Hull and White's rule without touching 2020. Those questions concern the rule's polynomial degree in δ and its 1/√T maturity scaling.

The design uses a 6-month trailing window, refitted monthly and tested on July–December. The main rehearsal year is 2019, with 2018 as a second year. The 2019-12-31 pairs end on 2020-01-02, so they are excluded.

The hedging error, on Hull and White's normalized scale, is e = (Δf − h·ΔS)/S_t:
- Δf is the option's mid-price change;
- ΔS is the index's change;
- h is the rule's hedge ratio.

The Gain of rule X over rule Y is 1 − Σe²_X / Σe²_Y. Intervals come from a circular block bootstrap over sessions, not options, because every option on a day shares that day's index move (§9).

In [1]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
R = ROOT / "results"
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True, "grid.alpha": 0.3})

In [2]:
g = pd.concat([pd.read_csv(R / f"rehearsal/gains_{y}.csv").assign(year=y) for y in (2019, 2018)])
show = g[g.rule.isin(["R1", "R2", "R3", "R4", "R5", "R6"]) & g.base.isin(["R0", "R1", "R3", "R4"])]
show[["year", "cp", "rule", "base", "gain", "lo", "hi", "pairs"]].round(4)

,year,cp,rule,base,gain,lo,hi,pairs
0,2019,C,R2,R1,0.2794,-0.0073,0.4213,127377
1,2019,C,R2,R3,0.0466,-0.0048,0.0850,127377
2,2019,C,R2,R4,0.1099,0.0182,0.1723,127377
3,2019,C,R1,R0,0.9478,0.9317,0.9641,127377
4,2019,C,R2,R0,0.9624,0.9519,0.9702,127377
5,2019,C,R4,R1,0.1904,-0.0426,0.3044,127377
6,2019,C,R3,R1,0.2442,-0.0256,0.3816,127377
7,2019,C,R5,R1,0.2814,-0.0051,0.4234,127377
8,2019,C,R6,R1,0.2436,-0.0159,0.3709,127377
16,2019,P,R2,R1,0.3350,0.1099,0.4231,172632


**Reading.** The Black–Scholes delta already removes about 95% of the unhedged squared error (R1 against R0).

In 2019 H2, Hull and White's correction (R2) removes a further 28% for calls and 34% for puts of what is left. It beats the fitted scaling rule (R4) by about 11% for both types. In 2018 H2, which contains the Q4 2018 sell-off, the call Gain is about zero and its 95% interval is about 0.48 wide; the put interval is about 0.29 wide.

That is the preview for 2020: a crash year will give wide intervals, and a correction fitted in calm months may not carry over.

## Specification check 1: the polynomial degree

Hull and White's adjustment is a quadratic in δ. Here degrees 0–3 are compared out of sample, on an orthogonal (Legendre) basis.

The criterion was fixed before running: the quadratic is *not contradicted* unless the cubic beats it, with a 95% bootstrap lower bound above 0.

In [3]:
s = pd.concat([pd.read_csv(R / f"rehearsal/spec_checks_{y}.csv").assign(year=y) for y in (2019, 2018)])
s[s.check.str.startswith("degree")].round(5)

,cp,check,gain,lo,p_adj,contradicts,year
0,C,degree 3 vs 2,0.00198,0.00010,0.01910,True,2019
5,P,degree 3 vs 2,0.00178,0.00002,0.02370,True,2019
0,C,degree 3 vs 2,0.00060,-0.00328,0.37816,False,2018
5,P,degree 3 vs 2,0.00257,-0.00147,0.14329,False,2018


In [4]:
g[g.base == "deg:2"][["year", "cp", "rule", "gain", "lo", "hi"]].round(4)

,year,cp,rule,gain,lo,hi
9,2019,C,deg:3,0.0020,0.0001,0.0042
10,2019,C,deg:1,0.0012,-0.0036,0.0048
11,2019,C,deg:0,0.0141,0.0007,0.0258
25,2019,P,deg:3,0.0018,0.0000,0.0032
26,2019,P,deg:1,-0.0002,-0.0017,0.0017
27,2019,P,deg:0,0.0001,-0.0051,0.0043
9,2018,C,deg:3,0.0006,-0.0033,0.0031
10,2018,C,deg:1,0.0013,-0.0016,0.0050
11,2018,C,deg:0,0.0182,-0.0038,0.0321
25,2018,P,deg:3,0.0026,-0.0015,0.0048


**Reading.** In 2019 the cubic beats the quadratic for both types, with a 95% lower bound above zero. Under the pre-specified diagnostic, this contradicts the quadratic. The improvement is statistically detectable with hundreds of thousands of pairs. Its economic size is negligible: about 0.2% of the squared error. In 2018 an improvement was not statistically established: the point Gains are positive, but the lower bounds are below zero.

Under the plan's rule, the cubic is carried to 2020 as a labelled exploratory rule; Hull and White's quadratic stays the rule under test.

A constant adjustment (degree 0) also beats the quadratic for calls in 2019, by 1.4%. The plan's criterion does not cover a lower degree winning, so this is reported, not acted on. It suggests the quadratic's extra coefficients add estimation noise on 6-month windows.

This was the first sign of a central finding: in the 2011–2023 walk-forward, one fitted constant gives essentially the same Gain as Hull and White's three coefficients (notebooks 08 and 09).

## Specification check 2: the 1/√T scaling

The test replaces τ^(−1/2) with τ^(−α), for α ∈ {0, 0.25, 0.75, 1}, each compared with α = ½. Holm's adjustment is applied across the four, per option type.

In [5]:
s[~s.check.str.startswith("degree")].round(4)

,cp,check,gain,lo,p_adj,contradicts,year
1,C,alpha:0 vs alpha:0.5,-0.2048,-0.3758,1.0000,False,2019
2,C,alpha:0.25 vs alpha:0.5,-0.0879,-0.1601,1.0000,False,2019
3,C,alpha:0.75 vs alpha:0.5,0.0264,0.0001,0.0972,False,2019
4,C,alpha:1 vs alpha:0.5,0.0064,-0.0160,0.8699,False,2019
6,P,alpha:0 vs alpha:0.5,-0.1537,-0.2392,1.0000,False,2019
7,P,alpha:0.25 vs alpha:0.5,-0.0478,-0.0799,1.0000,False,2019
8,P,alpha:0.75 vs alpha:0.5,-0.0184,-0.0321,1.0000,False,2019
9,P,alpha:1 vs alpha:0.5,-0.0715,-0.0995,1.0000,False,2019
1,C,alpha:0 vs alpha:0.5,-0.0420,-0.1957,1.0000,False,2018
2,C,alpha:0.25 vs alpha:0.5,-0.0266,-0.0993,1.0000,False,2018


**Reading.** The 1/√T scaling is not contradicted in either year. No scaling at all (α = 0) has a negative Gain against α = ½ for both types in both years (2019: −0.20 and −0.15; 2018: −0.04 and −0.04). The table shows only lower bounds, so the small 2018 gaps are not established.

For calls, α = ¾ (2019) and α = 1 (2018) look slightly better, but neither survives the adjustment. If anything, the call adjustment decays a little faster with maturity than 1/√T.

In [6]:
b = pd.read_csv(R / "rehearsal/buckets_2019.csv")
order = ["14-30", "31-60", "61-91", "92-122", "123-182", "183-365", "366+"]
t = b[b.pairs > 500].pivot_table(index=["cp", "delta_b"], columns="tau_b", values="sqrt_tau_kappa")[order]
t.round(3)

tau_b       14-30  31-60  61-91  92-122  123-182  183-365   366+
cp delta_b                                                      
C  0.1     -0.218 -0.246 -0.239  -0.224   -0.246   -0.213 -0.158
   0.2     -0.205 -0.226 -0.208  -0.179   -0.173   -0.130 -0.061
   0.3     -0.204 -0.209 -0.192  -0.137   -0.137   -0.088 -0.028
   0.4     -0.206 -0.206 -0.183  -0.157   -0.126   -0.079 -0.016
   0.5     -0.215 -0.214 -0.183  -0.153   -0.145   -0.103 -0.020
   0.6     -0.230 -0.226 -0.195  -0.143   -0.155   -0.107  0.006
   0.7     -0.240 -0.247 -0.182  -0.153   -0.177   -0.130  0.015
   0.8     -0.198 -0.197 -0.161  -0.149   -0.198   -0.099 -0.009
   0.9     -0.138 -0.150 -0.191  -0.008   -0.183   -0.021  0.192
P  0.1     -0.227 -0.246 -0.212  -0.189   -0.191   -0.202 -0.190
   0.2     -0.246 -0.264 -0.232  -0.206   -0.205   -0.205 -0.201
   0.3     -0.236 -0.249 -0.215  -0.192   -0.193   -0.191 -0.187
   0.4     -0.219 -0.231 -0.209  -0.173   -0.173   -0.172 -0.192
   0.5     -0.208 -0.222 -0.200  -0.172   -0.171   -0.155 -0.192
   0.6     -0.193 -0.214 -0.185  -0.178   -0.161   -0.153    NaN
   0.7     -0.174 -0.215 -0.208     NaN      NaN      NaN    NaN
   0.8     -0.198 -0.243    NaN     NaN      NaN      NaN    NaN
   0.9     -0.233 -0.208    NaN     NaN      NaN      NaN    NaN

This is the bucket analysis Hull and White describe in their §IV. Each cell is √τ_eff·κ̂, where κ̂ is the per-bucket no-intercept OLS coefficient of y = (Δf − δΔS)/S_t on w = (ν/S_t)·(ΔS/S_t), and τ_eff^(−1/2) = Σw²τ^(−1/2)/Σw². Under 1/√T, each row should be flat across maturity. For puts it is roughly flat. For calls the magnitude shrinks beyond six months, consistent with the slight preference for α > ½.

## Why the bootstrap is over sessions: naive against clustered standard errors

R2's coefficients are shown with ordinary OLS standard errors and with standard errors clustered by session.

In [7]:
pd.concat([pd.read_csv(R / f"rehearsal/se_ratio_{y}.csv").assign(year=y) for y in (2019, 2018)])[["year", "cp", "coef", "estimate", "se_naive", "se_clustered", "ratio", "sessions"]].round(4)

,year,cp,coef,estimate,se_naive,se_clustered,ratio,sessions
0,2019,C,a,-0.1450,0.0028,0.0411,14.5256,232
1,2019,C,b,0.1281,0.0125,0.0754,6.0483,232
2,2019,C,c,-0.1617,0.0127,0.1012,7.9585,232
3,2019,P,a,-0.1670,0.0020,0.0362,17.9262,232
4,2019,P,b,-0.0661,0.0100,0.0453,4.5174,232
5,2019,P,c,0.0436,0.0111,0.0716,6.4287,232
0,2018,C,a,-0.1409,0.0032,0.0744,23.3947,250
1,2018,C,b,0.0606,0.0147,0.1427,9.7231,250
2,2018,C,c,-0.1519,0.0155,0.1463,9.4282,250
3,2018,P,a,-0.3057,0.0026,0.0874,33.5290,250


**Reading.** The clustered standard errors are 4.5 to 34 times the naive ones. All options on a day share that day's index move and volatility shock, so the information accumulates with sessions (about 250 a year), not with option-days (about 300,000). Significance claims that treat options as independent, such as Hull and White's Appendix B, are therefore far too optimistic.

The same fact recurs later. It is why a fit on three options per session and type loses only about 0.01 in Gain (notebook 10), and why a tuned learner's correction shrinks almost to a constant (notebook 08).

## The dry run of the test script (2019)

After the rehearsal, the full test-year script (`scripts/run_test_year.py`) ran end to end with 2019 as the test year, on 12-month windows, excluding pairs whose day two is in 2020 (plan §15, 2026-11-04). It produces no 2020 number and is not evidence, since 2019 had already been used for the rehearsal. It tested the code, so that nothing would be debugged with 2020 results in view.

In [8]:
pd.read_csv(R / "dryrun_2019/verdict.csv").round(3)

,variant,verdict,H1,G_C,G_C_lo,G_C_hi,C_vs_R3,C_vs_R4,G_P,G_P_lo,G_P_hi,P_vs_R3,P_vs_R4
0,primary,Inconclusive,False,0.114,-0.052,0.239,not distinguished,not distinguished,0.099,-0.260,0.295,not distinguished,not distinguished
1,A,Inconclusive,False,0.085,-0.143,0.239,not distinguished,R2 beats,0.168,-0.121,0.333,not distinguished,not distinguished
2,B,Inconclusive,False,0.116,-0.046,0.239,not distinguished,not distinguished,0.091,-0.263,0.287,not distinguished,not distinguished
3,P-bid,Inconclusive,False,0.095,-0.009,0.182,not distinguished,R2 beats,0.064,-0.123,0.183,not distinguished,not distinguished
4,P-ask,Inconclusive,False,0.072,-0.067,0.191,not distinguished,not distinguished,0.087,-0.289,0.300,not distinguished,not distinguished
5,F,Inconclusive,False,0.161,-0.062,0.326,R2 beats,not distinguished,0.137,-0.292,0.377,not distinguished,not distinguished
6,W36,Inconclusive,False,0.157,-0.045,0.308,R2 beats,R2 beats,0.158,-0.201,0.360,not distinguished,not distinguished
7,W60,Inconclusive,False,0.159,-0.042,0.308,R2 beats,R2 beats,0.159,-0.194,0.358,not distinguished,not distinguished
8,S3,Inconclusive,False,0.120,-0.037,0.238,not distinguished,R2 beats,0.098,-0.256,0.294,not distinguished,not distinguished
9,endpoint low,Inconclusive,False,0.114,-0.052,0.239,not distinguished,not distinguished,0.099,-0.260,0.295,not distinguished,not distinguished


**Reading.** The script ran every variant of the verdict table. Even in 2019, a calm year, the procedure returns Inconclusive: calls 0.114 [−0.05, 0.24], puts 0.099 [−0.26, 0.30]. Those intervals are narrower than 2020's turned out to be (widths 0.29 and 0.56, against 0.82 and 0.90), but still too wide to establish a Gain of this size in one year. That is why the stability question is answered by the walk-forward over many years (notebook 06).